# tailguard — CVaR Reshoring Optimizer  ·  Full Extension Notebook
### Implements all extensions from *Cox-Process Scenario Generation, Decomposition, and Distributional Robustness for CVaR-Based Stochastic Reshoring* (2026)

**Pipeline:**
`GSCPI series` -> CIR calibration -> Cox generator -> (Sec.4) Stratified sampling -> (Sec.5) Benders -> (Sec.6) Spectral risk -> (Sec.7) Wasserstein DRO -> (Sec.8) Tail-dependence diagnostic -> (Sec.9) SAA validation

| Section | Extension | Status |
|---------|-----------|--------|
| Sec.3 | CIR + Cox process | original |
| Sec.4 | Stratified sampling on Lambda_T quantiles | new |
| Sec.5 | Benders decomposition of CVaR-MILP | new |
| Sec.6 | Spectral risk (multi-alpha CVaR blend) | new |
| Sec.7 | Wasserstein DRO (Mohajerin Esfahani-Kuhn) | new |
| Sec.8 | Tail-dependence diagnostic (lambda_U, Kendall tau) | new |
| Sec.9 | SAA validation protocol (Mak-Morton-Wood) | new |

In [ ]:
!pip install -q pulp ipywidgets requests openpyxl "xlrd>=2.0.1" scipy
import warnings; warnings.filterwarnings("ignore")
print("packages ready")

In [ ]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import pulp
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
np.random.seed(42)
print('imports done')

## Mathematical Framework

### Sec.2  Decision Problem (Chandran & Shanmuganathan 2026)

Choose one supplier per component $j \in \mathcal{J}$: $x_i \in \{0,1\}$.

$$Z^\omega = \sum_{j}\sum_{i \in \mathcal{I}_j} x_i\bigl(c_i + \kappa_i \cdot N_T^\omega\bigr)$$

Mean-plus-CVaR objective (Rockafellar-Uryasev):

$$\min_{x,\zeta,s}\; \bar{c}(x) + \lambda\!\left[\zeta + \frac{1}{(1-\alpha)\Omega}\sum_\omega s_\omega\right] \quad \text{s.t.}\; s_\omega \ge Z^\omega - \zeta,\; s_\omega \ge 0,\; \sum_{i\in\mathcal{I}_j}x_i=1$$

### Sec.3  CIR Intensity + Cox Process

$$d\lambda_t = a(b-\lambda_t)\,dt + \eta\sqrt{\lambda_t}\,dW_t, \qquad N_T|\Lambda_T \sim \text{Poisson}(\Lambda_T), \qquad \Lambda_T = \int_0^T \lambda_t\,dt$$

### Sec.4  Stratified Sampling (Eq. 15)

$$\widehat{\text{CVaR}}^{\text{strat}}_\alpha(Z) = \min_\zeta \left\{\zeta + \frac{1}{1-\alpha}\sum_{m=1}^M \frac{p_m}{\Omega_m}\sum_{\omega\in S_m}\max(0, Z^\omega - \zeta)\right\}$$

Variance reduction factor typically 4-10x (Lemma 4.1).

### Sec.5  Benders Decomposition

For fixed binary $\bar{x}$: $\zeta^\star = \text{VaR}_\alpha$, $s^\star_\omega = \max(0,Z^\omega-\zeta^\star)$. Master gets cut $\theta \ge \pi^{(\nu)\top}Z(x)$ each iteration.

### Sec.6  Spectral Risk (Eq. 22)

$$\rho_\phi(L) \approx \sum_{k=1}^K w_k\,\text{CVaR}_{\alpha_k}(L), \quad w_k\ge 0,\; \sum_k w_k=1$$

### Sec.7  Wasserstein DRO (Mohajerin Esfahani-Kuhn)

$$\sup_{Q\in\mathcal{B}_\varepsilon(\hat{P}_\Omega)}\text{CVaR}^Q_\alpha = \min_{\zeta,\tau\ge0,s\ge0}\;\zeta+\tau\varepsilon + \frac{1}{(1-\alpha)\Omega}\sum_\omega s_\omega \quad \text{s.t.}\; s_\omega\ge Z^\omega-\zeta$$

Radius $\varepsilon = c\cdot\Omega^{-1/2}$.


## Data Loading

| Mode | Requirement |
|------|-------------|
| **Synthetic** (default) | `data/synthetic/` folder |
| **Live** | FRED API key in env or widget |

In [ ]:
_fred_key_env = os.environ.get('FRED_API_KEY', '')

api_key_input = widgets.Password(
    value=_fred_key_env,
    placeholder='Paste FRED API key -- or leave blank to use synthetic data',
    description='FRED Key:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '80px'}
)

display(widgets.VBox([
    widgets.HTML('<b>FRED API key</b> -- leave blank to use synthetic CSV data'),
    widgets.HTML('<i style="color:#888">Get a free key at fredaccount.stlouisfed.org/apikeys</i>'),
    api_key_input,
]))
print('key widget ready')

In [ ]:
_gscpi_cache = {}
_SYNTH_PATH  = "data/synthetic"

# ── Built-in synthetic GSCPI (always available as last resort) ──────────
def _builtin_synthetic_gscpi():
    # AR(1) process mimicking GSCPI dynamics 2017-2026
    # Mean ~0, std ~1, autocorrelation ~0.85, COVID shock 2020-2021
    import hashlib
    rng = np.random.default_rng(20240101)
    dates = pd.date_range("2017-01-01", "2026-04-01", freq="MS")
    n = len(dates)
    v = np.zeros(n)
    for i in range(1, n):
        v[i] = 0.85 * v[i-1] + rng.normal(0, 0.4)
    # COVID shock: spike up 2020-03 to 2021-12
    covid = (dates >= "2020-03-01") & (dates <= "2021-12-01")
    v[covid] += np.linspace(1.5, 3.5, covid.sum()) * rng.uniform(0.7, 1.3, covid.sum())
    # Normalisation shock 2022
    norm = (dates >= "2022-01-01") & (dates <= "2022-10-01")
    v[norm] += np.linspace(3.0, -0.5, norm.sum())
    return pd.DataFrame({"date": dates, "value": np.round(v, 4)})

_BUILTIN_GSCPI = _builtin_synthetic_gscpi()

def _load_synthetic_gscpi(start, end):
    path = "{}/gscpi.csv".format(_SYNTH_PATH)
    df = pd.read_csv(path, parse_dates=["date"])
    df = df[(df["date"] >= start) & (df["date"] <= end)]
    return df[["date", "value"]].reset_index(drop=True)

def _parse_gscpi_bytes(raw):
    # Try openpyxl first (works for .xlsx and modern .xls)
    import io
    buf = io.BytesIO(raw)
    for engine in ["openpyxl", "xlrd"]:
        try:
            dfs = pd.read_excel(buf, sheet_name=None, header=None, engine=engine)
            # Find sheet with GSCPI data
            for sheet_name, df in dfs.items():
                if df.shape[1] >= 2:
                    data = df[[0, 1]].copy()
                    data.columns = ["date", "value"]
                    data = data[data["date"].notna() & (data["date"] != "Date")]
                    data["date"]  = pd.to_datetime(data["date"], dayfirst=True, errors="coerce")
                    data["value"] = pd.to_numeric(data["value"], errors="coerce")
                    data = data.dropna().sort_values("date").reset_index(drop=True)
                    if len(data) >= 12:
                        return data
            buf.seek(0)
        except Exception:
            buf = io.BytesIO(raw)
    raise ValueError("Could not parse GSCPI file with any engine")

def fetch_series(series_id, start, end):
    start_ts = pd.Timestamp(start)
    end_ts   = pd.Timestamp(end)
    if series_id == "GSCPI":
        # 1. Manual upload cache
        if _gscpi_cache.get("data") is not None:
            df = _gscpi_cache["data"].copy()
            df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)]
            if len(df) >= 3:
                print("  (manual upload: {} obs)".format(len(df)))
                return df.reset_index(drop=True)
        # 2. Synthetic CSV from disk
        try:
            df = _load_synthetic_gscpi(start, end)
            if len(df) >= 3:
                print("  (synthetic CSV: {} obs)".format(len(df)))
                return df
        except (FileNotFoundError, Exception):
            pass
        # 3. NY Fed live download (.xlsx, openpyxl)
        for url in [
            "https://www.newyorkfed.org/medialibrary/research/interactives/gscpi/downloads/gscpi_data.xlsx",
            "https://www.newyorkfed.org/medialibrary/research/interactives/gscpi/downloads/gscpi_data.xls",
        ]:
            try:
                r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0 (research)"})
                r.raise_for_status()
                if r.content[:4] in (b"<!DO", b"<htm", b"<HTM"):
                    continue
                df = _parse_gscpi_bytes(r.content)
                df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)]
                if len(df) >= 3:
                    _gscpi_cache["data"] = df.copy()
                    print("  (NY Fed live: {} obs)".format(len(df)))
                    return df.reset_index(drop=True)
            except Exception as e:
                print("  NY Fed {}: {}".format(url[-12:], str(e)[:60]))
        # 4. Built-in synthetic fallback (always works, no files needed)
        df = _BUILTIN_GSCPI.copy()
        df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)]
        print("  (built-in synthetic: {} obs -- CIR calibration will use simulated GSCPI)".format(len(df)))
        return df.reset_index(drop=True)
    # ── Non-GSCPI series ─────────────────────────────────────────────────
    try:
        df = pd.read_csv("{}/fred_series.csv".format(_SYNTH_PATH), parse_dates=["date"])
        df = df[(df["series_id"] == series_id) &
                (df["date"] >= start_ts) & (df["date"] <= end_ts)]
        if len(df) >= 3:
            print("  (synthetic CSV: {} {} obs)".format(series_id, len(df)))
            return df[["date", "value"]].reset_index(drop=True)
    except (FileNotFoundError, Exception):
        pass
    key = api_key_input.value.strip() or os.environ.get("FRED_API_KEY", "")
    if not key:
        # Synthetic fallback for non-GSCPI series too
        rng2 = np.random.default_rng(abs(hash(series_id)) % 2**31)
        dates2 = pd.date_range(start, end, freq="MS")
        vals2  = np.cumsum(rng2.normal(0, 0.3, len(dates2)))
        df2    = pd.DataFrame({"date": dates2, "value": np.round(vals2, 4)})
        print("  (synthetic fallback for {}: {} obs)".format(series_id, len(df2)))
        return df2
    r = requests.get("https://api.stlouisfed.org/fred/series/observations",
        params={"series_id": series_id, "api_key": key, "file_type": "json",
                "observation_start": start, "observation_end": end}, timeout=30)
    r.raise_for_status()
    obs = r.json().get("observations", [])
    if not obs:
        raise ValueError("No FRED data for {}".format(series_id))
    df = pd.DataFrame(obs)[["date", "value"]]
    df["date"]  = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna().reset_index(drop=True)

print("fetch_series ready  (built-in synthetic -> CSV -> NY Fed -> FRED)")

## Sec.3  Core Math: CIR · Cox · CVaR · BOM
*(Original implementation -- unchanged)*

## Your BOM Data

Upload your own Bill of Materials as CSV, or use the built-in Tesla example.

**CSV format:**
```
component,supplier,base_cost,kappa,lead_time,type
Battery_Cell,CATL_CN_sea,85,62,28,offshore
Battery_Cell,Panasonic_NV,128,4,1,domestic
```

| Column | Meaning |
|--------|---------|
| `component` | Part group (one row per sourcing option) |
| `supplier` | Supplier name / route label |
| `base_cost` | Cost per unit in normal times ($) |
| `kappa` | Extra cost per disruption event — calibrate from event studies |
| `lead_time` | Delivery days |
| `type` | `offshore` or `domestic` |

In [ ]:
# ── BOM Upload ──────────────────────────────────────────────────────────
import io as _io

BOM_DEFAULT = {
    "Battery_Cell": {
        "offshore": [("CATL_CN_sea",     85, 62, 28),
                     ("Panasonic_JP_sea", 95, 22, 21)],
        "domestic": [("Panasonic_NV",   128,  4)],
    },
    "Semiconductor": {
        "offshore": [("TSMC_TW_sea",    70, 85, 35),
                     ("Samsung_KR_air", 100, 28,  7)],
        "domestic": [("TSMC_AZ",        140,  5)],
    },
    "Structural": {
        "offshore": [("Giga_SH_sea",  55, 48, 30),
                     ("MX_truck",     65, 12,  4)],
        "domestic": [("Giga_TX",      90,  3)],
    },
    "Drive_Unit": {
        "offshore": [("CN_motor_sea",  120, 55, 32),
                     ("DE_motor_air",  145, 15,  5)],
        "domestic": [("Fremont",       175,  4)],
    },
    "Power_Electronics": {
        "offshore": [("TW_pcb_sea",  40, 70, 33),
                     ("KR_pcb_air",  58, 20,  6)],
        "domestic": [("Buffalo",     80,  3)],
    },
}

BOM = BOM_DEFAULT.copy()
_bom_source = "default (Tesla example)"


def _csv_to_bom(df):
    required = {"component", "supplier", "base_cost", "kappa", "lead_time", "type"}
    df.columns = df.columns.str.lower().str.strip()
    missing = required - set(df.columns)
    if missing:
        raise ValueError("Missing columns: {}".format(missing))
    df["base_cost"] = pd.to_numeric(df["base_cost"], errors="coerce")
    df["kappa"]     = pd.to_numeric(df["kappa"],     errors="coerce")
    df["lead_time"] = pd.to_numeric(df["lead_time"], errors="coerce").astype(int)
    df["type"]      = df["type"].str.strip().str.lower()
    if df[["base_cost","kappa","lead_time"]].isnull().any().any():
        raise ValueError("base_cost / kappa / lead_time must be numeric")
    bom = {}
    for comp, grp in df.groupby("component"):
        offshore, domestic = [], []
        for _, row in grp.iterrows():
            if row["type"] == "offshore":
                offshore.append((row["supplier"], int(row["base_cost"]),
                                 int(row["kappa"]), int(row["lead_time"])))
            else:
                domestic.append((row["supplier"], int(row["base_cost"]),
                                 int(row["kappa"])))
        if not offshore:
            raise ValueError("Component {} needs at least 1 offshore option".format(comp))
        if not domestic:
            raise ValueError("Component {} needs at least 1 domestic option".format(comp))
        bom[comp] = {"offshore": offshore, "domestic": domestic}
    return bom


def _show_bom_summary(bom, source):
    print("  Active BOM: {}".format(source))
    print("  {:<24} {:>6}  {:>6}  Suppliers".format("Component", "Offsh", "Dom"))
    print("  " + "-" * 62)
    for comp, d in bom.items():
        off_names = ", ".join(o[0] for o in d["offshore"])
        dom_names = ", ".join(o[0] for o in d["domestic"])
        print("  {:<24} {:>6}  {:>6}  {}  |  {}".format(
            comp, len(d["offshore"]), len(d["domestic"]),
            off_names[:22], dom_names[:22]))


bom_upload = widgets.FileUpload(accept=".csv", multiple=False,
               description="Upload BOM CSV",
               layout=widgets.Layout(width="220px"))
bom_reset  = widgets.Button(description="Reset to Tesla default",
               button_style="warning", layout=widgets.Layout(width="200px"))
bom_status = widgets.HTML()
bom_out    = widgets.Output()

def _on_bom_upload(change):
    global BOM, _bom_source
    with bom_out:
        clear_output(wait=True)
        try:
            up = bom_upload.value
            if not up: return
            item    = list(up.values())[0] if isinstance(up, dict) else up[0]
            content = item["content"] if isinstance(item, dict) else bytes(item["content"])
            df      = pd.read_csv(_io.BytesIO(content))
            BOM     = _csv_to_bom(df)
            _bom_source = "uploaded CSV ({} components)".format(len(BOM))
            bom_status.value = '<b style="color:#059669">Loaded: {}</b>'.format(_bom_source)
            _show_bom_summary(BOM, _bom_source)
        except Exception as e:
            bom_status.value = '<b style="color:#dc2626">Error: {}</b>'.format(e)
            print("Upload error: {}".format(e))

def _on_bom_reset(btn):
    global BOM, _bom_source
    with bom_out:
        clear_output(wait=True)
        BOM = BOM_DEFAULT.copy()
        _bom_source = "default (Tesla example)"
        bom_status.value = '<b style="color:#2563eb">Reset to Tesla default</b>'
        _show_bom_summary(BOM, _bom_source)

bom_upload.observe(_on_bom_upload, names="value")
bom_reset.on_click(_on_bom_reset)

display(widgets.VBox([
    widgets.HTML('<b style="font-size:14px">Bill of Materials</b>'),
    widgets.HTML('Upload your own CSV or run with Tesla default.'),
    widgets.HBox([bom_upload, bom_reset]),
    bom_status,
    bom_out,
]))
_show_bom_summary(BOM, _bom_source)
print("BOM widget ready")

In [ ]:
# ── Sec.3.1  CIR calibration ─────────────────────────────────────────────
def fit_cir(series):
    # OLS on Euler-Maruyama: Delta_lambda = a*b*dt - a*dt*lambda_{t-1} + eps
    # Returns (a, b, eta, lambda0, feller_satisfied)
    g = series.copy()
    g['lam'] = g['value'] - g['value'].min() + 0.1
    g['lam'] = g['lam'] / g['lam'].mean() * 0.5
    dt       = 1 / 12
    lam      = g['lam'].values
    dlam, lam_lag = np.diff(lam), lam[:-1]
    X    = np.vstack([np.ones_like(lam_lag), lam_lag]).T
    beta, *_ = np.linalg.lstsq(X, dlam, rcond=None)
    a    = max(0.1, -beta[1] / dt)
    b    = max(0.1,  beta[0] / (a * dt))
    resid = dlam - X @ beta
    eta  = np.sqrt(np.var(resid) / (np.mean(lam_lag) * dt))
    lam0 = g['lam'].iloc[-1]
    return a, b, eta, lam0, (2 * a * b >= eta ** 2)


# ── Sec.3.2  Cox process simulator ───────────────────────────────────────
def simulate_cox(a, b, eta, lam0, T, Omega, seed=42):
    # Euler-Maruyama CIR paths -> integrated intensity Lambda_T -> Poisson(Lambda_T)
    rng    = np.random.default_rng(seed)
    dt     = T / 252
    lam    = np.full(Omega, lam0, dtype=float)
    Lambda = np.zeros(Omega)
    N      = np.zeros(Omega, dtype=int)
    for _ in range(252):
        dW  = rng.standard_normal(Omega) * np.sqrt(dt)
        lam = np.maximum(
            lam + a * (b - lam) * dt + eta * np.sqrt(np.maximum(lam, 0)) * dW,
            1e-8)
        Lambda += lam * dt
        N      += rng.poisson(lam * dt)
    return N, Lambda


# ── Sec.3.3  Empirical CVaR ───────────────────────────────────────────────
def cvar(L, alpha):
    # E[L | L >= VaR_alpha(L)]
    s = np.sort(L)
    return s[int(np.floor(alpha * len(s))):].mean()


# ── BOM (bill of materials) ───────────────────────────────────────────────
BOM = {
    'Battery_Cell':  {
        'offshore': [('CATL_CN_sea',     85, 62, 28),
                     ('Panasonic_JP_sea', 95, 22, 21)],
        'domestic': [('Panasonic_NV',   128,  4)],
    },
    'Semiconductor': {
        'offshore': [('TSMC_TW_sea',    70, 85, 35),
                     ('Samsung_KR_air', 100, 28,  7)],
        'domestic': [('TSMC_AZ',        140,  5)],
    },
    'Structural': {
        'offshore': [('Giga_SH_sea',  55, 48, 30),
                     ('MX_truck',     65, 12,  4)],
        'domestic': [('Giga_TX',      90,  3)],
    },
    'Drive_Unit': {
        'offshore': [('CN_motor_sea',  120, 55, 32),
                     ('DE_motor_air',  145, 15,  5)],
        'domestic': [('Fremont',       175,  4)],
    },
    'Power_Electronics': {
        'offshore': [('TW_pcb_sea',  40, 70, 33),
                     ('KR_pcb_air',  58, 20,  6)],
        'domestic': [('Buffalo',     80,  3)],
    },
}

def _flatten_bom(bom):
    opts = []
    for comp, d in bom.items():
        for name, base, k, lt in d['offshore']:
            opts.append((comp, name, base, k, lt))
        for name, base, k in d['domestic']:
            opts.append((comp, name, base, k, 1))
    return opts


# ── Sec.2.4  CVaR-MILP (monolithic, original) ────────────────────────────
def solve_milp(bom, N_T, alpha=0.95, lam_w=1.0):
    # Rockafellar-Uryasev linearisation -- monolithic (all Omega slacks at once)
    Omega = len(N_T)
    opts  = _flatten_bom(bom)
    prob  = pulp.LpProblem('cvar_milp', pulp.LpMinimize)
    x     = {i: pulp.LpVariable(f'x{i}', cat='Binary')  for i in range(len(opts))}
    zeta  = pulp.LpVariable('zeta')
    s     = {w: pulp.LpVariable(f's{w}', lowBound=0)     for w in range(Omega)}
    for comp in bom:
        idx = [i for i, o in enumerate(opts) if o[0] == comp]
        prob += pulp.lpSum(x[i] for i in idx) == 1
    for w in range(Omega):
        Zw = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T[w]) for i in range(len(opts)))
        prob += s[w] >= Zw - zeta
    mean_c  = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T.mean()) for i in range(len(opts)))
    cvar_lp = zeta + pulp.lpSum(s[w] for w in range(Omega)) / ((1 - alpha) * Omega)
    prob   += mean_c + lam_w * cvar_lp
    solver  = (pulp.HiGHS_CMD(msg=False)
               if 'HiGHS_CMD' in pulp.listSolvers(True)
               else pulp.PULP_CBC_CMD(msg=False))
    prob.solve(solver)
    chosen = [opts[i] for i in range(len(opts)) if x[i].value() > 0.5]
    Z = np.array([sum(o[2] + o[3] * N_T[w] for o in chosen) for w in range(Omega)])
    return chosen, Z

print('Sec.3  CIR | Cox | CVaR-MILP (monolithic) -- loaded')


## Sec.4  Stratified Sampling on $\Lambda_T$ Quantiles

**Problem (Sec.4.1):** Crude MC allocates $\Omega$ scenarios uniformly; CVaR uses only the top $(1-\alpha)$ fraction.
Effective tail sample: $\Omega_e \approx \Omega/4$ to $\Omega/10$.

**Solution (Sec.4.2 / Eq. 15):** Partition $\Lambda_T$ support into $M$ strata, allocate $\Omega_m \propto p_m \sigma_m$ (Neyman), use weighted slacks.
LP structure **preserved** -- weights $p_m/\Omega_m$ are constants.


In [ ]:
# ── Sec.4  Stratified sampling on Lambda_T quantiles ─────────────────────

def simulate_cox_stratified(a, b, eta, lam0, T, Omega, M=10, seed=42):
    # Sec.4.2 -- Neyman-allocated acceptance-rejection sampling per stratum.
    # Returns (N_T, Lambda_T, pw) where pw[w] = p_m / Omega_m for scenario w.
    rng = np.random.default_rng(seed)

    # Pilot for quantile estimation
    pilot_size = max(Omega * 5, 20_000)
    _, lam_pilot = simulate_cox(a, b, eta, lam0, T, pilot_size, seed=seed + 1)

    quantiles   = np.linspace(0, 100, M + 1)
    breakpoints = np.percentile(lam_pilot, quantiles)
    breakpoints[0]  -= 1e-9
    breakpoints[-1] += 1e-9

    # Stratum probabilities and within-stratum sigma (proxy for cost sigma)
    p_m   = np.zeros(M)
    sig_m = np.zeros(M)
    for m in range(M):
        mask      = (lam_pilot >= breakpoints[m]) & (lam_pilot < breakpoints[m + 1])
        p_m[m]    = mask.mean()
        sig_m[m]  = lam_pilot[mask].std() if mask.sum() > 1 else 1e-6

    # Neyman allocation: Omega_m proportional to p_m * sigma_m
    alloc   = p_m * sig_m
    alloc   = alloc / alloc.sum()
    Omega_m = np.maximum(np.round(alloc * Omega).astype(int), 1)
    Omega_m[-1] = Omega - Omega_m[:-1].sum()

    N_all, L_all, pw_all = [], [], []
    for m in range(M):
        budget = Omega_m[m]
        lo, hi = breakpoints[m], breakpoints[m + 1]
        w      = p_m[m] / budget
        coll_N, coll_L = [], []
        batch = max(budget * 20, 500)
        while len(coll_N) < budget:
            N_b, L_b = simulate_cox(a, b, eta, lam0, T, int(batch),
                                    seed=int(rng.integers(0, 2**31)))
            mask = (L_b >= lo) & (L_b < hi)
            coll_N.extend(N_b[mask].tolist())
            coll_L.extend(L_b[mask].tolist())
        N_all.extend(coll_N[:budget])
        L_all.extend(coll_L[:budget])
        pw_all.extend([w] * budget)

    return (np.array(N_all, dtype=int),
            np.array(L_all, dtype=float),
            np.array(pw_all, dtype=float))


def cvar_stratified(Z, pw, alpha):
    # Sec.4 weighted empirical CVaR (Eq. 15)
    order  = np.argsort(Z)
    Z_s    = Z[order];  pw_s = pw[order]
    cum_w  = np.cumsum(pw_s); cum_w /= cum_w[-1]
    idx    = min(np.searchsorted(cum_w, alpha), len(Z_s) - 1)
    zeta   = Z_s[idx]
    tail   = Z_s >= zeta
    denom  = pw_s[tail].sum()
    return float((pw_s[tail] * Z_s[tail]).sum() / denom) if denom > 0 else float(zeta)


def solve_milp_stratified(bom, N_T, pw, alpha=0.95, lam_w=1.0):
    # Sec.4 -- CVaR-MILP with stratum-weighted slacks.
    # Objective: mean_c(x) + lambda * [zeta + 1/(1-alpha) * sum_w pw_norm_w * s_w]
    Omega   = len(N_T)
    opts    = _flatten_bom(bom)
    prob    = pulp.LpProblem('cvar_milp_strat', pulp.LpMinimize)
    x       = {i: pulp.LpVariable(f'x{i}', cat='Binary') for i in range(len(opts))}
    zeta    = pulp.LpVariable('zeta')
    s       = {w: pulp.LpVariable(f's{w}', lowBound=0)   for w in range(Omega)}
    for comp in bom:
        idx = [i for i, o in enumerate(opts) if o[0] == comp]
        prob += pulp.lpSum(x[i] for i in idx) == 1
    pw_norm = pw / pw.sum()
    for w in range(Omega):
        Zw = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T[w]) for i in range(len(opts)))
        prob += s[w] >= Zw - zeta
    mean_c  = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T.mean()) for i in range(len(opts)))
    cvar_lp = zeta + pulp.lpSum(pw_norm[w] * s[w] for w in range(Omega)) / (1 - alpha)
    prob   += mean_c + lam_w * cvar_lp
    solver  = (pulp.HiGHS_CMD(msg=False) if 'HiGHS_CMD' in pulp.listSolvers(True)
               else pulp.PULP_CBC_CMD(msg=False))
    prob.solve(solver)
    chosen = [opts[i] for i in range(len(opts)) if x[i].value() > 0.5]
    Z = np.array([sum(o[2] + o[3] * N_T[w] for o in chosen) for w in range(Omega)])
    return chosen, Z

print('Sec.4  Stratified sampling + weighted CVaR-MILP -- loaded')


## Sec.5  Benders Decomposition of the CVaR-MILP

**Block structure:** For fixed $\bar{x}$, CVaR subproblem is pure LP with closed-form solution.

**Cuts:** $\theta \ge \pi^{(\nu)\top}Z(x)$ added to master each iteration.

**Convergence (Prop.5.1):** Finite, since $\Pi$ has finitely many extreme points.


In [ ]:
# ── Sec.5  Benders decomposition ─────────────────────────────────────────

def _subproblem_cf(Z_omega, alpha):
    # Sec.5.1 closed-form CVaR subproblem:
    # zeta* = VaR_alpha,  s*_w = max(0, Z_w - zeta*)
    # phi*  = zeta* + 1/((1-alpha)*Omega) * sum s*_w
    zeta_star = float(np.percentile(Z_omega, 100 * alpha))
    s_star    = np.maximum(Z_omega - zeta_star, 0.0)
    phi_star  = zeta_star + s_star.sum() / ((1 - alpha) * len(Z_omega))
    return zeta_star, s_star, phi_star


def _benders_cut(opts, s_star, N_T):
    # Sec.5.2 -- dual multipliers mu_w = exceedance / n_exceed (uniform on tail)
    # cut coefficient coeff[i] = sum_w mu_w * (c_i + kappa_i * N_w)
    exceedance = (s_star > 0).astype(float)
    n_exceed   = exceedance.sum()
    mu = exceedance / n_exceed if n_exceed > 0 else np.zeros(len(N_T))
    return np.array([float(np.sum(mu * (opts[i][2] + opts[i][3] * N_T)))
                     for i in range(len(opts))])


def solve_milp_benders(bom, N_T, alpha=0.95, lam_w=1.0,
                       max_iter=60, tol=1e-3, verbose=True):
    # Sec.5 Algorithm 1 -- Benders decomposition.
    # Master: min mean_c(x) + lambda*theta  s.t. cuts theta >= pi^T Z(x)
    # Subproblem: closed-form phi(x_bar) generates one cut per iteration.
    opts   = _flatten_bom(bom)
    n_opts = len(opts)
    cuts   = []
    best_x = None;  UB = float('inf');  LB = -float('inf')

    if verbose:
        print(f"{'Iter':>4}  {'LB':>12}  {'UB':>12}  {'Gap%':>8}  {'Cuts':>5}")
        print('-' * 48)

    for nu in range(max_iter):
        # Master MILP
        master = pulp.LpProblem(f'bm_{nu}', pulp.LpMinimize)
        x      = {i: pulp.LpVariable(f'x{i}', cat='Binary') for i in range(n_opts)}
        theta  = pulp.LpVariable('theta', lowBound=-1e8)
        for comp in bom:
            idx = [i for i, o in enumerate(opts) if o[0] == comp]
            master += pulp.lpSum(x[i] for i in idx) == 1
        for coeff, rhs in cuts:
            master += theta >= pulp.lpSum(coeff[i] * x[i] for i in range(n_opts)) + rhs
        mean_c = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T.mean())
                            for i in range(n_opts))
        master += mean_c + lam_w * theta
        solver  = (pulp.HiGHS_CMD(msg=False) if 'HiGHS_CMD' in pulp.listSolvers(True)
                   else pulp.PULP_CBC_CMD(msg=False))
        master.solve(solver)
        if pulp.LpStatus[master.status] != 'Optimal':
            if verbose: print(f'  Master infeasible at iter {nu}')
            break
        LB = pulp.value(master.objective)
        chosen_idx = [i for i in range(n_opts) if x[i].value() > 0.5]

        # Subproblem (closed form)
        Z_omega    = np.array([sum(opts[i][2] + opts[i][3] * N_T[w] for i in chosen_idx)
                               for w in range(len(N_T))])
        mean_c_val = sum(opts[i][2] + opts[i][3] * N_T.mean() for i in chosen_idx)
        zeta_star, s_star, phi_star = _subproblem_cf(Z_omega, alpha)

        UB_cand = mean_c_val + lam_w * phi_star
        if UB_cand < UB:
            UB = UB_cand;  best_x = chosen_idx

        gap = (UB - LB) / (abs(UB) + 1e-9)
        if verbose:
            print(f'{nu:>4}  {LB:>12.2f}  {UB:>12.2f}  {gap*100:>7.3f}%  {len(cuts):>5}')
        if gap < tol:
            if verbose: print(f'  Converged (gap {gap*100:.4f}% < {tol*100}%)')
            break

        # Generate optimality cut
        coeff = _benders_cut(opts, s_star, N_T)
        rhs   = phi_star - float(sum(coeff[i] * (1 if i in chosen_idx else 0)
                                     for i in range(n_opts)))
        cuts.append((coeff, rhs))

    chosen = [opts[i] for i in best_x]
    Z = np.array([sum(o[2] + o[3] * N_T[w] for o in chosen) for w in range(len(N_T))])
    return chosen, Z

print('Sec.5  Benders decomposition -- loaded')


## Sec.6  Spectral Risk Generalization

$\text{CVaR}_\alpha$ is the step kernel $\phi(u)=\mathbf{1}\{u\ge\alpha\}/(1-\alpha)$.  
Smooth kernels embed the planner's preferences across the full quantile range.

$K$ CVaR levels introduce $K$ aux. variables $\zeta_k$ and $K\Omega$ slacks -- program remains MILP.


In [ ]:
# ── Sec.6  Spectral risk generalization ──────────────────────────────────

SPECTRAL_KERNELS = {
    'CVaR (step, alpha=0.95)': {
        'alphas': [0.95], 'weights': [1.0],
        'desc': 'Standard CVaR at alpha=0.95',
    },
    'Exponential (beta=4)': {
        'alphas': [0.80, 0.90, 0.95, 0.975, 0.99],
        'weights': None, 'beta': 4.0,
        'desc': 'Smooth upweighting of extreme quantiles',
    },
    'Linear (phi=2u)': {
        'alphas': [0.50, 0.70, 0.85, 0.93, 0.98],
        'weights': None,
        'desc': 'Linear kernel -- equal weight from median to worst',
    },
    'Wang blend': {
        'alphas': [0.90, 0.95, 0.99],
        'weights': [0.25, 0.45, 0.30],
        'desc': 'Ad-hoc blend emphasising extreme tail',
    },
}

def _spectral_weights(cfg):
    alphas = cfg['alphas']
    w = cfg.get('weights')
    if w is not None:
        arr = np.array(w, dtype=float); return alphas, (arr / arr.sum()).tolist()
    if 'beta' in cfg:
        arr = np.exp(cfg['beta'] * np.array(alphas)); return alphas, (arr / arr.sum()).tolist()
    # Linear: phi(u) = 2u
    arr = 2 * np.array(alphas, dtype=float); return alphas, (arr / arr.sum()).tolist()


def cvar_spectral(Z, kernel='CVaR (step, alpha=0.95)'):
    # Eq.22 -- weighted sum of CVaRs
    cfg = SPECTRAL_KERNELS[kernel]
    alphas, weights = _spectral_weights(cfg)
    return sum(w * cvar(Z, a) for w, a in zip(weights, alphas))


def solve_milp_spectral(bom, N_T, kernel='CVaR (step, alpha=0.95)', lam_w=1.0):
    # Sec.6 Lemma 6.1 -- K sets of slacks s_{w,k} and VaR surrogates zeta_k.
    # Objective: mean_c(x) + lambda * sum_k w_k [zeta_k + 1/((1-alpha_k)*Omega) * sum_w s_{w,k}]
    cfg            = SPECTRAL_KERNELS[kernel]
    alphas, weights = _spectral_weights(cfg)
    K      = len(alphas)
    Omega  = len(N_T)
    opts   = _flatten_bom(bom)
    prob   = pulp.LpProblem('cvar_spectral', pulp.LpMinimize)
    x      = {i: pulp.LpVariable(f'x{i}', cat='Binary') for i in range(len(opts))}
    zeta   = {k: pulp.LpVariable(f'z{k}') for k in range(K)}
    s      = {(w, k): pulp.LpVariable(f's{w}_{k}', lowBound=0)
              for w in range(Omega) for k in range(K)}
    for comp in bom:
        idx = [i for i, o in enumerate(opts) if o[0] == comp]
        prob += pulp.lpSum(x[i] for i in idx) == 1
    for k in range(K):
        for w in range(Omega):
            Zw = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T[w]) for i in range(len(opts)))
            prob += s[(w, k)] >= Zw - zeta[k]
    mean_c = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T.mean()) for i in range(len(opts)))
    spectral = pulp.lpSum(
        weights[k] * (zeta[k] + pulp.lpSum(s[(w,k)] for w in range(Omega)) / ((1-alphas[k])*Omega))
        for k in range(K))
    prob += mean_c + lam_w * spectral
    solver = (pulp.HiGHS_CMD(msg=False) if 'HiGHS_CMD' in pulp.listSolvers(True)
              else pulp.PULP_CBC_CMD(msg=False))
    prob.solve(solver)
    chosen = [opts[i] for i in range(len(opts)) if x[i].value() > 0.5]
    Z = np.array([sum(o[2] + o[3] * N_T[w] for o in chosen) for w in range(Omega)])
    return chosen, Z

print('Sec.6  Spectral risk generalization -- loaded')


## Sec.7  Wasserstein DRO

**Calibration uncertainty** in $(a,b,\eta)$ propagates to CVaR with bias $O\!\left((1-\alpha)^{-1/2}\Omega^{-1/2}\right)$.

Single extra dual variable $\tau\ge0$, penalty $\tau\varepsilon$ -- **no new combinatorial structure**.
Radius $\varepsilon = c/\sqrt{\Omega}$ (concentration of measure, Sec.7.4).


In [ ]:
# ── Sec.7  Wasserstein DRO ───────────────────────────────────────────────

def solve_milp_dro(bom, N_T, alpha=0.95, lam_w=1.0, epsilon=None, c_radius=1.0):
    # Sec.7 Theorem 7.1 -- adds scalar tau >= 0 and penalty tau*epsilon.
    # Objective: mean_c(x) + lambda * [zeta + tau*epsilon + 1/((1-alpha)*Omega) * sum s_w]
    # epsilon default: c / sqrt(Omega)  (Sec.7.4 concentration guarantee)
    Omega = len(N_T)
    if epsilon is None:
        epsilon = c_radius / math.sqrt(Omega)
    opts  = _flatten_bom(bom)
    prob  = pulp.LpProblem('cvar_dro', pulp.LpMinimize)
    x     = {i: pulp.LpVariable(f'x{i}', cat='Binary') for i in range(len(opts))}
    zeta  = pulp.LpVariable('zeta')
    tau   = pulp.LpVariable('tau', lowBound=0)
    s     = {w: pulp.LpVariable(f's{w}', lowBound=0) for w in range(Omega)}
    for comp in bom:
        idx = [i for i, o in enumerate(opts) if o[0] == comp]
        prob += pulp.lpSum(x[i] for i in idx) == 1
    for w in range(Omega):
        Zw = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T[w]) for i in range(len(opts)))
        prob += s[w] >= Zw - zeta
    mean_c   = pulp.lpSum(x[i] * (opts[i][2] + opts[i][3] * N_T.mean()) for i in range(len(opts)))
    dro_cvar = zeta + tau * epsilon + pulp.lpSum(s[w] for w in range(Omega)) / ((1 - alpha) * Omega)
    prob    += mean_c + lam_w * dro_cvar
    solver   = (pulp.HiGHS_CMD(msg=False) if 'HiGHS_CMD' in pulp.listSolvers(True)
                else pulp.PULP_CBC_CMD(msg=False))
    prob.solve(solver)
    tau_val = tau.value()
    chosen  = [opts[i] for i in range(len(opts)) if x[i].value() > 0.5]
    Z       = np.array([sum(o[2] + o[3] * N_T[w] for o in chosen) for w in range(Omega)])
    return chosen, Z, tau_val, epsilon


def dro_radius_sensitivity(bom, N_T, alpha=0.95, lam_w=1.0,
                            c_values=None):
    # Vary DRO radius c (epsilon = c/sqrt(Omega)) and tabulate results
    if c_values is None:
        c_values = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]
    rows = []
    for c in c_values:
        chosen, Z, tau_val, eps = solve_milp_dro(bom, N_T, alpha=alpha, lam_w=lam_w, c_radius=c)
        dom_pct = sum(1 for o in chosen if o[4] == 1) / len(chosen) * 100
        rows.append({
            'c': c, 'epsilon': round(eps, 6),
            'tau*': round(tau_val or 0.0, 4),
            'Mean cost': round(Z.mean(), 2),
            f'CVaR_{alpha:.0%}': round(cvar(Z, alpha), 2),
            'Domestic %': round(dom_pct, 1),
        })
    return pd.DataFrame(rows)

print('Sec.7  Wasserstein DRO -- loaded')


## Sec.8  Tail-Dependence Diagnostic

**Proposition 3.2:** Under independent shocks $\lambda_U^{\text{indep}}=0$; under Cox (shared $N_T$), $\lambda_U^{\text{Cox}}>0$.

| Metric | Independent | Cox (expected) |
|--------|-------------|----------------|
| $P(X_1>q_\alpha \mid X_2>q_\alpha)$ | $\approx 1-\alpha$ | $\gg 1-\alpha$ |
| Kendall $\tau$ (joint tail) | $\approx 0$ | $>0$ |
| $\lambda_U$ at $u=0.99$ | $\approx 0$ | $>0$ |


In [ ]:
# -- Sec.8  Tail-dependence diagnostic --

def tail_dependence_analysis(a, b, eta, lam0, T=1.0, Omega=10_000,
                              kappa1=80, kappa2=25, base1=100, base2=135,
                              alpha=0.95, seed=42):
    results = {}
    N_shared, _ = simulate_cox(a, b, eta, lam0, T, Omega, seed=seed)
    N_indep1, _ = simulate_cox(a, b, eta, lam0, T, Omega, seed=seed)
    N_indep2, _ = simulate_cox(a, b, eta, lam0, T, Omega, seed=seed + 999)
    generators = {"Independent": (N_indep1, N_indep2), "Cox (shared)": (N_shared, N_shared.copy())}
    for name, (N1, N2) in generators.items():
        X1 = base1 + kappa1 * N1.astype(float)
        X2 = base2 + kappa2 * N2.astype(float)
        q1 = np.percentile(X1, 100 * alpha)
        q2 = np.percentile(X2, 100 * alpha)
        tail1 = X1 > q1;  tail2 = X2 > q2;  joint = tail1 & tail2
        p_cond = joint.sum() / max(tail2.sum(), 1)
        tau_val = 0.0
        if joint.sum() >= 10:
            tau_val, _ = stats.kendalltau(X1[joint], X2[joint])
        us = np.linspace(0.80, 0.99, 20)
        lam_u = []
        for u in us:
            q1u = np.percentile(X1, 100 * u)
            q2u = np.percentile(X2, 100 * u)
            denom = (X2 > q2u).sum()
            lam_u.append(((X1 > q1u) & (X2 > q2u)).sum() / max(denom, 1))
        results[name] = {
            "X1": X1, "X2": X2, "q1": q1, "q2": q2,
            "p_conditional": p_cond, "kendall_tau": tau_val,
            "lambda_u_curve": (us, np.array(lam_u)),
        }
    return results


def plot_tail_dependence(results, alpha=0.95):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    colors = {"Independent": "#2563eb", "Cox (shared)": "#dc2626"}
    for idx, (name, res) in enumerate(results.items()):
        ax = axes[idx]
        X1, X2 = res["X1"], res["X2"]
        q1, q2 = res["q1"], res["q2"]
        jm = (X1 > q1) & (X2 > q2)
        ax.scatter(X1[~jm], X2[~jm], s=3, alpha=0.2, color=colors[name])
        ax.scatter(X1[jm], X2[jm], s=14, alpha=0.85, color="#f59e0b",
                   label="joint tail (n={})".format(jm.sum()))
        ax.axvline(q1, color="gray", ls="--", lw=1)
        ax.axhline(q2, color="gray", ls="--", lw=1)
        ax.set_xlabel("Route 1 cost ($)")
        ax.set_ylabel("Route 2 cost ($)")
        ax.set_title("{}  P={:.3f}  tau={:.3f}".format(name, res["p_conditional"], res["kendall_tau"]))
        ax.legend(fontsize=8, markerscale=2)
    for name, res in results.items():
        us, lam_u = res["lambda_u_curve"]
        mk = "o" if name == "Independent" else "s"
        axes[2].plot(us, lam_u, lw=2.5, color=colors[name], marker=mk, markersize=4, label=name)
    axes[2].axhline(1 - alpha, color="gray", ls=":", lw=1.5, label="baseline={:.2f}".format(1 - alpha))
    axes[2].set_xlabel("Threshold u")
    axes[2].set_ylabel("lambda_U(u)")
    axes[2].set_title("Upper tail dependence (Prop 3.2)")
    axes[2].legend(fontsize=9)
    plt.suptitle("Sec.8.5 Tail-Dependence Diagnostic", fontsize=12, fontweight="bold")
    plt.tight_layout();  plt.show()
    ri = results["Independent"];  rc = results["Cox (shared)"]
    print("  {:<42} {:>8}  {:>8}".format("Metric", "Indep", "Cox"))
    print("  " + "-" * 58)
    print("  {:<42} {:>8.3f}  {:>8.3f}".format("P(X1>q|X2>q)", ri["p_conditional"], rc["p_conditional"]))
    print("  {:<42} {:>8.3f}  {:>8.3f}".format("Kendall tau (joint tail)", ri["kendall_tau"], rc["kendall_tau"]))
    print("  {:<42} {:>8.3f}  {:>8.3f}".format("lambda_U at u=0.99", ri["lambda_u_curve"][1][-1], rc["lambda_u_curve"][1][-1]))

print("Sec.8  Tail-dependence diagnostic -- loaded")


## Sec.9  SAA Validation Protocol (Mak-Morton-Wood 1999)

1. **Lower bound** $M$ independent SAA replications of size $\Omega$:
   $\text{LB} = \bar{v} - t_{M-1,0.025}\cdot s_v/\sqrt{M}$
2. **Upper bound** evaluate $\hat{x}$ from rep 0 on $\Omega'\gg\Omega$ fresh scenarios:
   $\text{UB} = \hat{v}(\Omega') + C(1-\alpha)^{-1/2}(\Omega')^{-1/2}$
3. **Optimality gap** $\text{UB}-\text{LB}$. Below 1% is acceptable.


In [ ]:
# -- Sec.9  SAA Validation Protocol (Mak-Morton-Wood) --

def saa_validation(bom, a, b, eta, lam0, T=1.0, alpha=0.95, lam_w=1.0,
                   M=10, Omega=1000, Omega_prime=20_000, seed_base=0, verbose=True):
    obj_vals = [];  first_chosen = None
    if verbose:
        print("[ Sec.9 SAA ]  M={}  Omega={}  Omega_prime={}  alpha={:.2f}".format(M, Omega, Omega_prime, alpha))
    for m in range(M):
        N_m, _ = simulate_cox(a, b, eta, lam0, T, Omega, seed=seed_base + m)
        chosen_m, Z_m = solve_milp(bom, N_m, alpha=alpha, lam_w=lam_w)
        obj_m = Z_m.mean() + lam_w * cvar(Z_m, alpha)
        obj_vals.append(obj_m)
        if m == 0:  first_chosen = chosen_m
        if verbose:
            dom_pct = sum(1 for o in chosen_m if o[4]==1) / len(chosen_m) * 100
            print("  Rep {:>2}: obj={:8.2f}  domestic={:.0f}%".format(m, obj_m, dom_pct))
    v_bar  = np.mean(obj_vals)
    s_v    = np.std(obj_vals, ddof=1)
    t_crit = stats.t.ppf(0.975, df=M - 1)
    LB     = v_bar - t_crit * s_v / math.sqrt(M)
    N_eval, _ = simulate_cox(a, b, eta, lam0, T, Omega_prime, seed=seed_base + 10000)
    Z_eval = np.array([sum(o[2] + o[3] * N_eval[w] for o in first_chosen) for w in range(Omega_prime)])
    correction = 1.0 / ((1 - alpha) ** 0.5 * Omega_prime ** 0.5)
    UB = Z_eval.mean() + lam_w * (cvar(Z_eval, alpha) + correction)
    gap = UB - LB;  gap_pct = gap / abs(UB) * 100
    if verbose:
        print()
        print("  LB  = {:10.3f}  v_bar={:.3f}".format(LB, v_bar))
        print("  UB  = {:10.3f}  correction={:.4f}".format(UB, correction))
        gap_label = "OK <1%" if gap_pct < 1 else "WARNING >1%"
        print("  Gap = {:10.3f}  ({:.3f}%  {})".format(gap, gap_pct, gap_label))
        print()
        dom_pct = sum(1 for o in first_chosen if o[4]==1) / len(first_chosen) * 100
        print("  Optimal decision (rep 0):  domestic={:.0f}%".format(dom_pct))
        for o in first_chosen:
            tag = "domestic" if o[4] == 1 else "offshore(LT={}d)".format(o[4])
            print("  {:<22} {:<22} {:<18} {}".format(o[0], o[1], tag, o[3]))
        print("  Mean cost (eval): ${:.2f}".format(Z_eval.mean()))
        print("  CVaR (eval): ${:.2f}".format(cvar(Z_eval, alpha)))
    return {"LB": LB, "UB": UB, "gap": gap, "gap_pct": gap_pct,
            "v_bar": v_bar, "s_v": s_v, "UB_correction": correction,
            "chosen_first": first_chosen, "Z_eval": Z_eval, "obj_vals": obj_vals}


def plot_saa_results(saa_res, alpha=0.95):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    objs = saa_res["obj_vals"];  M = len(objs)
    axes[0].bar(range(M), objs, color="#2563eb", alpha=0.7, edgecolor="white")
    axes[0].axhline(saa_res["v_bar"], color="#059669", lw=2, ls="--", label="v_bar={:.2f}".format(saa_res["v_bar"]))
    axes[0].axhline(saa_res["LB"],   color="#dc2626", lw=1.5, ls=":", label="LB={:.2f}".format(saa_res["LB"]))
    axes[0].axhline(saa_res["UB"],   color="#f59e0b", lw=1.5, ls=":", label="UB={:.2f}".format(saa_res["UB"]))
    axes[0].set_xlabel("SAA Replication");  axes[0].set_ylabel("Objective")
    axes[0].set_title("Sec.9.2 SAA (M={})  Gap={:.2f}%".format(M, saa_res["gap_pct"]), fontweight="bold")
    axes[0].legend(fontsize=9)
    Z_eval = saa_res["Z_eval"]
    axes[1].hist(Z_eval, bins=60, color="#059669", edgecolor="white", alpha=0.8)
    axes[1].axvline(Z_eval.mean(), color="#2563eb", lw=2, ls="--", label="Mean=${:.0f}".format(Z_eval.mean()))
    axes[1].axvline(cvar(Z_eval, alpha), color="#dc2626", lw=2, ls="--", label="CVaR=${:.0f}".format(cvar(Z_eval, alpha)))
    axes[1].set_xlabel("Cost ($)")
    axes[1].set_title("OOS Distribution (Omega_prime={:,})".format(len(Z_eval)), fontweight="bold")
    axes[1].legend(fontsize=9)
    plt.suptitle("Sec.9  SAA Validation Protocol", fontsize=12, fontweight="bold")
    plt.tight_layout();  plt.show()

print("Sec.9  SAA validation protocol -- loaded")


## Control Panel: Window Configuration & Run

In [ ]:
PRESETS = {
    '🟢 Pre-shock baseline  (2017-2019 -> 2020-2022)': {
        'train': ('2017-01-01', '2019-12-31'),
        'test':  ('2020-01-01', '2022-06-30'),
        'desc':  'Blind to COVID + chip shortage.',
    },
    '🟡 Include COVID ramp  (2017-2021 -> 2022)': {
        'train': ('2017-01-01', '2021-12-31'),
        'test':  ('2022-01-01', '2022-12-31'),
        'desc':  'Blind to 2022 chip shortage.',
    },
    '🟠 Include shock       (2017-2022 -> 2023-2024)': {
        'train': ('2017-01-01', '2022-12-31'),
        'test':  ('2023-01-01', '2024-12-31'),
        'desc':  'Blind to normalisation.',
    },
    '🔴 Full history        (2017-2024 -> 2025-2026)': {
        'train': ('2017-01-01', '2024-12-31'),
        'test':  ('2025-01-01', '2026-04-01'),
        'desc':  'Blind to IEEPA tariff regime.',
    },
    'Custom': {
        'train': ('2017-01-01', '2020-12-31'),
        'test':  ('2021-01-01', '2023-12-31'),
        'desc':  'Set your own dates.',
    },
}
ALL_DATES = pd.date_range('2017-01-01', '2026-07-01', freq='QS')
DATE_OPTS  = [d.strftime('%Y-%m-%d') for d in ALL_DATES]

def _snap(ds):
    ts = pd.Timestamp(ds)
    return DATE_OPTS[min(range(len(DATE_OPTS)), key=lambda i: abs(pd.Timestamp(DATE_OPTS[i]) - ts))]

preset_dd    = widgets.Dropdown(options=list(PRESETS.keys()),
                 description='Preset:', layout=widgets.Layout(width='540px'),
                 style={'description_width': '60px'})
desc_label   = widgets.HTML()
train_start  = widgets.Dropdown(options=DATE_OPTS, value='2017-01-01',
                 description='Train from:', style={'description_width': '90px'},
                 layout=widgets.Layout(width='260px'))
train_end    = widgets.Dropdown(options=DATE_OPTS, value='2019-10-01',
                 description='Train to:', style={'description_width': '90px'},
                 layout=widgets.Layout(width='260px'))
test_start   = widgets.Dropdown(options=DATE_OPTS, value='2020-01-01',
                 description='Test from:', style={'description_width': '90px'},
                 layout=widgets.Layout(width='260px'))
test_end     = widgets.Dropdown(options=DATE_OPTS, value='2022-04-01',
                 description='Test to:', style={'description_width': '90px'},
                 layout=widgets.Layout(width='260px'))
intensity_dd = widgets.Dropdown(
    options=[
        ('GSCPI -- Global Supply Chain Pressure (NY Fed)', 'GSCPI'),
        ('IR14110 -- Import Price: Deep Sea Freight', 'IR14110'),
        ('CES3000000003 -- Mfg Hourly Earnings', 'CES3000000003'),
    ],
    value='GSCPI', description='Intensity:',
    style={'description_width': '80px'}, layout=widgets.Layout(width='520px'))
alpha_slider  = widgets.FloatSlider(value=0.95, min=0.80, max=0.99, step=0.01,
    description='CVaR alpha:', readout_format='.2f',
    style={'description_width': '80px'}, layout=widgets.Layout(width='380px'))
lambda_slider = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.25,
    description='lambda:', readout_format='.2f',
    style={'description_width': '70px'}, layout=widgets.Layout(width='380px'))
omega_dd = widgets.Dropdown(
    options=[('500 (debug)', 500), ('1,000 (fast)', 1000),
             ('2,000 (rec.)', 2000), ('5,000', 5000)],
    value=1000, description='Scenarios Omega:',
    style={'description_width': '100px'}, layout=widgets.Layout(width='280px'))
solver_checks = widgets.SelectMultiple(
    options=['Sec.3 Monolithic (original)', 'Sec.4 Stratified sampling',
             'Sec.5 Benders decomposition', 'Sec.6 Spectral risk',
             'Sec.7 Wasserstein DRO', 'Sec.8 Tail-dependence diagnostic',
             'Sec.9 SAA validation protocol'],
    value=['Sec.3 Monolithic (original)', 'Sec.4 Stratified sampling',
           'Sec.5 Benders decomposition', 'Sec.6 Spectral risk',
           'Sec.7 Wasserstein DRO', 'Sec.8 Tail-dependence diagnostic',
           'Sec.9 SAA validation protocol'],
    description='Run (Ctrl+click):', layout=widgets.Layout(width='540px', height='165px'),
    style={'description_width': '100px'})
spectral_dd = widgets.Dropdown(
    options=list(SPECTRAL_KERNELS.keys()), value='Exponential (beta=4)',
    description='Kernel (Sec.6):', style={'description_width': '90px'},
    layout=widgets.Layout(width='420px'))
dro_c_slider = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.5,
    description='DRO c (Sec.7):', readout_format='.1f',
    style={'description_width': '90px'}, layout=widgets.Layout(width='380px'))
saa_M_dd = widgets.Dropdown(
    options=[('M=5', 5), ('M=10', 10), ('M=20', 20), ('M=30', 30)],
    value=5, description='SAA M (Sec.9):',
    style={'description_width': '100px'}, layout=widgets.Layout(width='260px'))
run_btn = widgets.Button(description='Run selected extensions',
    button_style='success', layout=widgets.Layout(width='260px', height='44px'))
output  = widgets.Output()

def on_preset_change(change):
    p = PRESETS[change["new"]]
    train_start.value = _snap(p["train"][0]);  train_end.value = _snap(p["train"][1])
    test_start.value  = _snap(p["test"][0]);   test_end.value  = _snap(p["test"][1])
    desc_label.value  = f'<i style="color:#555">{p["desc"]}</i>'

preset_dd.observe(on_preset_change, names="value")
on_preset_change({"new": preset_dd.value})

display(widgets.VBox([
    widgets.HTML('<b style="font-size:15px">Window + Solver Configuration</b>'),
    preset_dd, desc_label,
    widgets.HTML('<hr style="margin:6px 0">'),
    widgets.HTML('<b>Training window</b>'),
    widgets.HBox([train_start, train_end]),
    widgets.HTML('<b>Test window</b>'),
    widgets.HBox([test_start, test_end]),
    widgets.HTML('<hr style="margin:6px 0">'),
    intensity_dd, widgets.HBox([alpha_slider, lambda_slider]), omega_dd,
    widgets.HTML('<hr style="margin:6px 0">'),
    widgets.HTML('<b>Extensions (Ctrl+click to select/deselect)</b>'),
    solver_checks,
    spectral_dd, widgets.HBox([dro_c_slider, saa_M_dd]),
    widgets.HTML('<hr style="margin:6px 0">'),
    run_btn, output,
]))
print('Configuration widget ready')


In [ ]:
def _print_sel(label, chosen):
    print("  {}:".format(label))
    for o in chosen:
        tag = "domestic" if o[4] == 1 else "offshore(LT={}d)".format(o[4])
        print("    [{}] {:<22} {:<24} kappa={}".format(tag, o[0], o[1], o[3]))

def run_all(btn):
    with output:
        clear_output(wait=True)
        ts    = train_start.value
        te    = train_end.value
        vs    = test_start.value
        ve    = test_end.value
        sid   = intensity_dd.value
        alpha = alpha_slider.value
        lam_w = lambda_slider.value
        Omega = omega_dd.value
        sel   = set(solver_checks.value)
        if pd.Timestamp(te) >= pd.Timestamp(vs):
            print("Train end must be before test start."); return
        print("[ Data ] fetching {}...".format(sid))
        try:
            train_df = fetch_series(sid, ts, te)
            test_df  = fetch_series(sid, vs, ve)
        except Exception as e:
            print("Error: {}".format(e)); return
        print("  Train: {} obs  |  Test: {} obs".format(len(train_df), len(test_df)))
        if len(train_df) < 12:
            print("Need >= 12 training obs."); return
        a, b, eta, lam0, feller = fit_cir(train_df)
        feller_str = "OK" if feller else "VIOLATED"
        print("[ Sec.3 CIR ]  a={:.4f}  b={:.4f}  eta={:.4f}  lambda0={:.4f}".format(a, b, eta, lam0))
        print("  Feller 2ab={:.3f}  eta^2={:.3f}  {}".format(2*a*b, eta**2, feller_str))
        N_T, Lambda_T = simulate_cox(a, b, eta, lam0, 1.0, Omega)
        overdisp = N_T.var() / max(N_T.mean(), 1e-9)
        cox_str = "Cox justified" if overdisp > 1.05 else "near-Poisson"
        print("[ Sec.3 Cox ]  E[N]={:.3f}  Var={:.3f}  OD={:.2f}  ({})".format(
            N_T.mean(), N_T.var(), overdisp, cox_str))
        if "Sec.3 Monolithic (original)" in sel:
            print("\n[ Sec.3 MILP ]  alpha={}  lambda={}  Omega={}".format(alpha, lam_w, Omega))
            chosen_m, Z_m = solve_milp(BOM, N_T, alpha=alpha, lam_w=lam_w)
            dom_m = sum(1 for o in chosen_m if o[4]==1) / len(chosen_m) * 100
            print("  Mean=${:.1f}  CVaR={:.1f}  Domestic={:.0f}%".format(
                Z_m.mean(), cvar(Z_m, alpha), dom_m))
            _print_sel("Sec.3 Monolithic", chosen_m)
        if "Sec.4 Stratified sampling" in sel:
            print("\n[ Sec.4 Stratified ]  M=10 strata  Omega={}...".format(Omega))
            N_s, L_s, pw_s = simulate_cox_stratified(a, b, eta, lam0, 1.0, Omega, M=10)
            chosen_s, Z_s  = solve_milp_stratified(BOM, N_s, pw_s, alpha=alpha, lam_w=lam_w)
            dom_s  = sum(1 for o in chosen_s if o[4]==1) / len(chosen_s) * 100
            cvar_s = cvar_stratified(Z_s, pw_s, alpha)
            print("  Mean=${:.1f}  CVaR(weighted)={:.1f}  Domestic={:.0f}%".format(
                Z_s.mean(), cvar_s, dom_s))
            _print_sel("Sec.4 Stratified", chosen_s)
        if "Sec.5 Benders decomposition" in sel:
            print("\n[ Sec.5 Benders ]  max_iter=40  tol=0.5%...")
            chosen_b, Z_b = solve_milp_benders(
                BOM, N_T, alpha=alpha, lam_w=lam_w, max_iter=40, tol=0.005, verbose=True)
            dom_b = sum(1 for o in chosen_b if o[4]==1) / len(chosen_b) * 100
            print("  Mean=${:.1f}  CVaR={:.1f}  Domestic={:.0f}%".format(
                Z_b.mean(), cvar(Z_b, alpha), dom_b))
            _print_sel("Sec.5 Benders", chosen_b)
        if "Sec.6 Spectral risk" in sel:
            kernel = spectral_dd.value
            print("\n[ Sec.6 Spectral ]  kernel={}...".format(kernel))
            chosen_sp, Z_sp = solve_milp_spectral(BOM, N_T, kernel=kernel, lam_w=lam_w)
            dom_sp = sum(1 for o in chosen_sp if o[4]==1) / len(chosen_sp) * 100
            rho_sp = cvar_spectral(Z_sp, kernel)
            print("  Mean=${:.1f}  rho_phi={:.1f}  Domestic={:.0f}%".format(
                Z_sp.mean(), rho_sp, dom_sp))
            _print_sel("Sec.6 Spectral", chosen_sp)
        if "Sec.7 Wasserstein DRO" in sel:
            c_val = dro_c_slider.value
            eps_preview = c_val / Omega**0.5
            print("\n[ Sec.7 DRO ]  c={}  epsilon={:.5f}...".format(c_val, eps_preview))
            chosen_d, Z_d, tau_star, eps = solve_milp_dro(
                BOM, N_T, alpha=alpha, lam_w=lam_w, c_radius=c_val)
            dom_d = sum(1 for o in chosen_d if o[4]==1) / len(chosen_d) * 100
            print("  eps={:.5f}  tau*={:.4f}  Mean=${:.1f}  CVaR={:.1f}  Dom={:.0f}%".format(
                eps, tau_star or 0.0, Z_d.mean(), cvar(Z_d, alpha), dom_d))
            _print_sel("Sec.7 DRO", chosen_d)
            print("\n  DRO radius sensitivity:")
            print(dro_radius_sensitivity(BOM, N_T, alpha=alpha, lam_w=lam_w).to_string(index=False))
        if "Sec.8 Tail-dependence diagnostic" in sel:
            print("\n[ Sec.8 Tail dependence ]  Omega={}...".format(Omega))
            td_res = tail_dependence_analysis(a, b, eta, lam0, Omega=Omega, alpha=alpha)
            plot_tail_dependence(td_res, alpha=alpha)
        if "Sec.9 SAA validation protocol" in sel:
            M_saa       = saa_M_dd.value
            Omega_saa   = min(Omega, 500)
            Omega_prime = min(Omega * 10, 5000)
            print("\n[ Sec.9 SAA ]  M={}  Omega={}  Omega_prime={}...".format(
                M_saa, Omega_saa, Omega_prime))
            saa_res = saa_validation(BOM, a, b, eta, lam0, alpha=alpha, lam_w=lam_w,
                                     M=M_saa, Omega=Omega_saa, Omega_prime=Omega_prime, verbose=True)
            plot_saa_results(saa_res, alpha=alpha)
        print("\n[ Summary ] plotting...")
        train_q95  = float(np.percentile(train_df["value"].dropna(), 95))
        exceedance = (test_df["value"] > train_q95).mean()
        fig = plt.figure(figsize=(14, 9))
        gs  = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.35)
        ax1 = fig.add_subplot(gs[0, :])
        ax1.plot(train_df["date"], train_df["value"], color="#2563eb", lw=2, label="Train")
        ax1.plot(test_df["date"],  test_df["value"],  color="#dc2626", lw=2, ls="--", label="Test")
        ax1.axhline(train_q95, color="#f59e0b", ls=":", lw=1.5,
                    label="Train q95={:.2f}".format(train_q95))
        ax1.axvspan(pd.Timestamp(vs), pd.Timestamp(ve), alpha=0.08, color="red")
        ax1.set_title("{} -- exceedance={:.0%}".format(sid, exceedance), fontweight="bold")
        ax1.legend(fontsize=9)
        ax2 = fig.add_subplot(gs[1, 0])
        ax2.hist(N_T, bins=range(0, min(int(N_T.max())+2, 22)),
                 color="#2563eb", edgecolor="white", alpha=0.85)
        ax2.axvline(N_T.mean(), color="#dc2626", ls="--",
                    label="E[N]={:.2f}".format(N_T.mean()))
        ax2.set_title("Sec.3 Cox distribution N_T", fontweight="bold")
        ax2.set_xlabel("Shock count");  ax2.legend(fontsize=9)
        ax3 = fig.add_subplot(gs[1, 1])
        try:
            c_p, Z_p = solve_milp(BOM, N_T, alpha=alpha, lam_w=lam_w)
            ax3.hist(Z_p, bins=50, color="#059669", edgecolor="white", alpha=0.85)
            ax3.axvline(Z_p.mean(), color="#2563eb", ls="--",
                        label="Mean=${:.0f}".format(Z_p.mean()))
            ax3.axvline(cvar(Z_p, alpha), color="#dc2626", ls="--",
                        label="CVaR=${:.0f}".format(cvar(Z_p, alpha)))
            ax3.set_title("Optimal cost distribution Z", fontweight="bold")
            ax3.set_xlabel("Total cost ($)");  ax3.legend(fontsize=9)
        except Exception:
            pass
        plt.suptitle("Train: {} -> {}  Test: {} -> {}  alpha={:.2f}  lambda={}  Omega={:,}".format(
            ts, te, vs, ve, alpha, lam_w, Omega), fontsize=10)
        plt.show()

run_btn.on_click(run_all)
print("Run handler wired -- press the button to start")


## Efficient Frontier -- lambda Sweep

Vary $\lambda$ from 0 to 10. Uses window + intensity from control panel above.

In [ ]:
sweep_btn    = widgets.Button(description="Run lambda sweep",
    button_style="warning", layout=widgets.Layout(width="200px"))
sweep_output = widgets.Output()

def run_sweep(btn):
    with sweep_output:
        clear_output(wait=True)
        sid   = intensity_dd.value
        ts    = train_start.value
        te    = train_end.value
        alpha = alpha_slider.value
        Omega = omega_dd.value
        print("Fetching {}...".format(sid))
        try:
            train_df = fetch_series(sid, ts, te)
        except Exception as e:
            print("Error: {}".format(e)); return
        a, b, eta, lam0, _ = fit_cir(train_df)
        N_T, _ = simulate_cox(a, b, eta, lam0, 1.0, Omega)
        lambdas = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
        rows = []
        for lw in lambdas:
            print("  lambda={:.2f} ...".format(lw), end=" ", flush=True)
            chosen, Z = solve_milp(BOM, N_T, alpha=alpha, lam_w=lw)
            dom = sum(1 for o in chosen if o[4] == 1) / len(chosen) * 100
            cvar_key = "CVaR_{:.0%}".format(alpha)
            rows.append({"lambda": lw, "mean": Z.mean(), cvar_key: cvar(Z, alpha), "dom_%": dom})
            print("done")
        fdf = pd.DataFrame(rows)
        print()
        print(fdf.to_string(index=False))
        cvar_key = "CVaR_{:.0%}".format(alpha)
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        axes[0].plot(fdf["mean"], fdf[cvar_key], "o-", color="#2563eb", lw=2, markersize=9)
        for _, r in fdf.iterrows():
            axes[0].annotate("l={:.2f}".format(r["lambda"]), (r["mean"], r[cvar_key]),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
        axes[0].set_xlabel("Mean cost ($)")
        axes[0].set_ylabel("CVaR (alpha={:.0%})".format(alpha))
        axes[0].set_title("Efficient frontier", fontweight="bold")
        axes[1].plot(fdf["lambda"], fdf["dom_%"], "s-", color="#059669", lw=2, markersize=9)
        axes[1].set_xlabel("lambda"); axes[1].set_ylabel("Domestic %")
        axes[1].set_title("Domestic share vs risk aversion", fontweight="bold")
        plt.tight_layout(); plt.show()

sweep_btn.on_click(run_sweep)
display(widgets.VBox([sweep_btn, sweep_output]))
